<div style="border: 2px solid #4CAF50; padding: 10px; border-radius: 8px; background-color: #e8f5e9; text-align: center;">
  <strong style="font-size: 32px;">REDES NEURONALES CONVOLUCIONALES</strong><br>
  <strong style="font-size: 24px;">Keras/Tensorflow</strong>
</div>

# Ejemplo CNN

## Descripción del problema

- **Base de datos**: MNIST (60,000 imágenes de entrenamiento y 10,000 de prueba).  
- **Formato de imagen**: 28×28 píxeles en escala de grises.  
- **Objetivo**: Identificar a qué dígito (0–9) corresponde cada imagen.  

---

## Arquitectura CNN típica para MNIST

Una CNN sencilla podría incluir:

- **Capa convolucional**:  
  - Filtros de 3×3 (9 parámetros).
  - Cada filtro tiene un parámetro adicional de sesgo .
  - Detecta bordes, esquinas y patrones locales.  

- **Capa de pooling (max-pooling 2×2)**:  
  - Reduce la dimensión espacial (submuestreo).  
  - Mantiene características relevantes.  

- **Segunda convolución + pooling**:  
  - Aprende patrones más complejos (curvas, formas).  

- **Capa totalmente conectada (fully connected)**:  
  - Combina las características para tomar la decisión.  

- **Softmax**:  
  - Devuelve probabilidades para las 10 clases.  


In [1]:
# ============================================
# Importar librerías necesarias
# ============================================
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.utils import plot_model
import pydot

In [3]:
# ============================================
# Cargar y preparar los datos
# ============================================
# MNIST trae imágenes de dígitos escritos a mano (0-9)
(x_train, y_train), (x_test, y_test) = datasets.mnist.load_data()

# Normalizamos los valores de píxeles (0-255) a rango [0,1]
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Keras espera un canal (grayscale), así que añadimos la dimensión del canal
x_train = x_train.reshape((-1, 28, 28, 1))
x_test = x_test.reshape((-1, 28, 28, 1))

print("Forma de los datos de entrenamiento:", x_train.shape)
print("Forma de los datos de prueba:", x_test.shape)

Forma de los datos de entrenamiento: (60000, 28, 28, 1)
Forma de los datos de prueba: (10000, 28, 28, 1)


In [4]:
# ============================================
# Construir la CNN
# ============================================
model = models.Sequential([

    # Capa convolucional: 32 filtros de 3x3
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)), 
    #input_shape=(), define el tamaño de la entrada para poder construir la red.
    #Parámetros de la capa: F = 3, S = 1 (por defecto), P = 0 (valid) (por defecto).
    
    # Pooling para reducir tamaño (stride=2 por defecto)
    layers.MaxPooling2D(pool_size=(2,2)),
    #Parámetros de la capa: F = 2, S = 2 (por defecto), P = 0 (valid) (por defecto).

    # Segunda convolución con más profundidad
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(pool_size=(2,2)),

    # Pasamos a vector plano
    layers.Flatten(),

    # Capa totalmente conectada (oculta)
    layers.Dense(64, activation='relu'),

    # Capa de salida: 10 neuronas (una por cada dígito)
    layers.Dense(10, activation='softmax')
])

C:\Users\Flavio\anaconda3\envs\Keras-TF_GPU\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


## Cálculo del número de parámetros, profundidad y tamaño de salida en cada capa de la CNN

En una convolución o *pooling*, el tamaño de la salida en ancho y alto se calcula como:

$$
W_{out} = \left\lfloor \frac{W_{in} - F + 2P}{S} \right\rfloor + 1, 
\quad
H_{out} = \left\lfloor \frac{H_{in} - F + 2P}{S} \right\rfloor + 1
$$

donde:  
- $W_{in}, H_{in}$ son el ancho y alto de la entrada.  
- $F$ es el tamaño del filtro o ventana (*kernel* o *pool size*).  
- $P$ es el *padding*.  
- $S$ es el *stride*.  
- La **profundidad de salida** depende del número de filtros en una convolución, o se mantiene igual en *pooling*.  

---

### Primera convolución: `Conv2D(32, kernel_size=(3,3))`
- **Entrada**: $28 \times 28 \times 1$  
- **Parámetros de la capa**: $F = 3$, $S = 1$, $P = 0$ (*valid*).  
- **Cálculo del tamaño**:
  $$
  W_{out} = H_{out} = \left\lfloor \frac{28 - 3 + 0}{1} \right\rfloor + 1 = 26
  $$
  Profundidad = número de filtros = 32.  

**Salida**: $26 \times 26 \times 32$  
**Parámetros**:
$$
( \text{Tamaño del filtro (ventana)} + \text{(bias)} ) \times \text{(Número de filtros)} 
$$
$$
\big( (3 \times 3 \times 1) + 1 \big) \times 32 = 320
$$

---

### Primera capa de pooling: `MaxPooling2D(pool_size=(2,2))`
- **Entrada**: $26 \times 26 \times 32$  
- **Parámetros de la capa**: $F = 2$, $S = 2$, $P = 0$.  
- **Cálculo del tamaño**:
  $$
  W_{out} = H_{out} = \left\lfloor \frac{26 - 2 + 0}{2} \right\rfloor + 1 = 13
  $$
  Profundidad = 32 (no cambia).  

**Salida**: $13 \times 13 \times 32$  
**Parámetros**: 0  

---

### Segunda convolución: `Conv2D(64, kernel_size=(3,3))`
- **Entrada**: $13 \times 13 \times 32$  
- **Parámetros de la capa**: $F = 3$, $S = 1$, $P = 0$.  
- **Cálculo del tamaño**:
  $$
  W_{out} = H_{out} = \left\lfloor \frac{13 - 3 + 0}{1} \right\rfloor + 1 = 11
  $$
  Profundidad = número de filtros = 64.  

**Salida**: $11 \times 11 \times 64$  
**Parámetros**:
$$
( \text{Tamaño del filtro (ventana)} + \text{(bias)} ) \times \text{(Número de filtros)} 
$$
$$
\big( (3 \times 3 \times 32) + 1 \big) \times 64 = 18.496
$$

---

### Segunda capa de pooling: `MaxPooling2D(pool_size=(2,2))`
- **Entrada**: $11 \times 11 \times 64$  
- **Parámetros de la capa**: $F = 2$, $S = 2$, $P = 0$.  
- **Cálculo del tamaño**:
  $$
  W_{out} = H_{out} = \left\lfloor \frac{11 - 2 + 0}{2} \right\rfloor + 1 = 5
  $$
  Profundidad = 64.  

**Salida**: $5 \times 5 \times 64$  
**Parámetros**: 0  

---

### Capa de aplanamiento: `Flatten()`
- **Entrada**: $5 \times 5 \times 64$  
- **Operación**: convierte el tensor tridimensional en un vector unidimensional:  
  $$
  5 \times 5 \times 64 = 1600
  $$  

**Salida**: 1600  
**Parámetros**: 0 (solo reorganiza datos, no entrena nada).  

---

### Capa totalmente conectada: `Dense(64)`
- **Entrada**: 1600  
- **Cálculo del tamaño**:  
  Al conectar todas las entradas a 64 neuronas, la salida es un vector de 64.  

**Salida**: 64  
**Parámetros**:
$$
(1600 \times 64) + 64 \ (\text{bias}) = 102.464
$$

---

### Capa de salida: `Dense(10)`
- **Entrada**: 64  
- **Cálculo del tamaño**:  
  Cada neurona representa una clase (dígito 0–9), por lo tanto:  
  $$
  \text{Salida} = 10
  $$  

**Salida**: 10  
**Parámetros**:
$$
(64 \times 10) + 10 \ (\text{bias}) = 650
$$

---

In [5]:
# ============================================
# Mostrar arquitectura
# ============================================
model.summary()

#plot_model(model, to_file="fig/Latex/cnn_Keras_architecture.png", show_shapes=True, show_layer_names=True)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 26, 26, 32)          │             320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 13, 13, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 11, 11, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 5, 5, 64)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 1600)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 64)                  │         102,464 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 10)                  │             650 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 121,930 (476.29 KB)

 Trainable params: 121,930 (476.29 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
# ============================================
# Compilar el modelo
# ============================================
# - Optimizador: Adam (ajusta los pesos automáticamente)
# - Función de pérdida: entropía cruzada para clasificación multiclase
# - Métrica: exactitud
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

In [7]:
# ============================================
# Entrenar el modelo
# ============================================
history = model.fit(
    x_train, y_train,
    epochs=5,              # número de pasadas completas por los datos
    batch_size=64,         # cuántas imágenes procesa antes de actualizar pesos
    validation_data=(x_test, y_test), # validación en datos de prueba
    verbose=2              # 2 = mostrar progreso compacto
)

Epoch 1/5
938/938 - 9s - 9ms/step - accuracy: 0.9494 - loss: 0.1728 - val_accuracy: 0.9802 - val_loss: 0.0604
Epoch 2/5
938/938 - 8s - 8ms/step - accuracy: 0.9823 - loss: 0.0565 - val_accuracy: 0.9762 - val_loss: 0.0749
Epoch 3/5
938/938 - 7s - 8ms/step - accuracy: 0.9878 - loss: 0.0390 - val_accuracy: 0.9895 - val_loss: 0.0318
Epoch 4/5
938/938 - 7s - 8ms/step - accuracy: 0.9911 - loss: 0.0290 - val_accuracy: 0.9896 - val_loss: 0.0330
Epoch 5/5
938/938 - 7s - 8ms/step - accuracy: 0.9929 - loss: 0.0228 - val_accuracy: 0.9924 - val_loss: 0.0252


In [8]:
# ============================================
# Evaluar el modelo
# ============================================
print("EVALUACION\n")
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"\nExactitud en prueba: {test_acc:.2f}")

EVALUACION


Exactitud en prueba: 0.99


In [9]:
# ============================================
# Hacer predicciones
# ============================================
import numpy as np

# Tomamos una imagen de prueba
sample = x_test[0].reshape(1,28,28,1)
prediction = model.predict(sample)

print("\nProbabilidades por clase:", prediction)
print("Predicción final:", np.argmax(prediction))
print("Etiqueta real:", y_test[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step

Probabilidades por clase: [[2.2040958e-09 1.3401873e-07 6.2267527e-06 6.7713401e-08 2.9252707e-09
  9.3451531e-12 3.0208068e-13 9.9999297e-01 5.1735881e-08 4.2519719e-07]]
Predicción final: 7
Etiqueta real: 7


## Dropout en Redes Neuronales

### Uso en el código
En Keras/TensorFlow, **se implementa como una capa más dentro del modelo:**

- layers.Dropout(0.25),   # Apaga aleatoriamente 25% de neuronas
- layers.Dropout(0.5),    # Apaga 50% de las conexiones en entrenamiento


In [10]:
# ============================================
# Construir la CNN con Dropout
# ============================================
model = models.Sequential([

    # Capa convolucional: 32 filtros de 3x3
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)), 
    # input_shape=() define el tamaño de la entrada para poder construir la red.
    # Parámetros de la capa: F = 3, S = 1 (por defecto), P = 0 (valid).

    # Pooling para reducir tamaño (stride=2 por defecto)
    layers.MaxPooling2D(pool_size=(2,2)),
    # Parámetros de la capa: F = 2, S = 2 (por defecto), P = 0 (valid).

    # Dropout después del primer bloque conv+pooling
    layers.Dropout(0.25),   # Apaga aleatoriamente 25% de neuronas

    # Segunda convolución con más profundidad
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(pool_size=(2,2)),

    # Otro Dropout después del segundo bloque conv+pooling
    layers.Dropout(0.25),

    # Pasamos a vector plano
    layers.Flatten(),

    # Capa totalmente conectada (oculta)
    layers.Dense(64, activation='relu'),

    # Dropout antes de la capa de salida
    layers.Dropout(0.5),    # Apaga 50% de las conexiones en entrenamiento

    # Capa de salida: 10 neuronas (una por cada dígito)
    layers.Dense(10, activation='softmax')
])


# ============================================
# Mostrar arquitectura
# ============================================
model.summary()

#plot_model(model, to_file="fig/Latex/cnn_Keras-D_architecture.png", show_shapes=True, show_layer_names=True)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)                    │ (None, 26, 26, 32)          │             320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 13, 13, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 13, 13, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 11, 11, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_3 (MaxPooling2D)       │ (None, 5, 5, 64)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 5, 5, 64)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_1 (Flatten)                  │ (None, 1600)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 64)                  │         102,464 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 10)                  │             650 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 121,930 (476.29 KB)

 Trainable params: 121,930 (476.29 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
# ============================================
# Compilar el modelo
# ============================================
# - Optimizador: Adam (ajusta los pesos automáticamente)
# - Función de pérdida: entropía cruzada para clasificación multiclase
# - Métrica: exactitud
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])


# ============================================
# Entrenar el modelo
# ============================================
history = model.fit(
    x_train, y_train,
    epochs=5,              # número de pasadas completas por los datos
    batch_size=64,         # cuántas imágenes procesa antes de actualizar pesos
    validation_data=(x_test, y_test), # validación en datos de prueba
    verbose=2              # 2 = mostrar progreso compacto
)

# ============================================
# Evaluar el modelo
# ============================================
print("EVALUACION\n")
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"\nExactitud en prueba: {test_acc:.2f}")

Epoch 1/5
938/938 - 10s - 11ms/step - accuracy: 0.8834 - loss: 0.3743 - val_accuracy: 0.9788 - val_loss: 0.0733
Epoch 2/5
938/938 - 9s - 9ms/step - accuracy: 0.9565 - loss: 0.1447 - val_accuracy: 0.9852 - val_loss: 0.0460
Epoch 3/5
938/938 - 9s - 9ms/step - accuracy: 0.9673 - loss: 0.1095 - val_accuracy: 0.9872 - val_loss: 0.0396
Epoch 4/5
938/938 - 9s - 10ms/step - accuracy: 0.9723 - loss: 0.0943 - val_accuracy: 0.9874 - val_loss: 0.0336
Epoch 5/5
938/938 - 9s - 10ms/step - accuracy: 0.9754 - loss: 0.0827 - val_accuracy: 0.9885 - val_loss: 0.0336
EVALUACION


Exactitud en prueba: 0.99


In [12]:
# ============================================
# Hacer predicciones
# ============================================
import numpy as np

# Tomamos una imagen de prueba
sample = x_test[0].reshape(1,28,28,1)
prediction = model.predict(sample)

print("\nProbabilidades por clase:", prediction)
print("Predicción final:", np.argmax(prediction))
print("Etiqueta real:", y_test[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step

Probabilidades por clase: [[2.1933953e-11 3.7388792e-10 1.3355716e-06 4.9400622e-08 4.8762594e-10
  3.0355761e-12 5.4956329e-18 9.9999857e-01 1.7874344e-10 7.1333915e-08]]
Predicción final: 7
Etiqueta real: 7


## Evaluacion

#### Predicciones

In [13]:
y_pred = model.predict(x_test)
y_pred = y_pred.argmax(axis=1)
y_true = y_test

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


### Matriz de confusión

In [14]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred)
print(cm)

[[ 976    0    1    0    0    0    1    1    1    0]
 [   0 1131    2    2    0    0    0    0    0    0]
 [   1    0 1025    2    0    0    0    4    0    0]
 [   0    0    2 1003    0    3    0    1    1    0]
 [   1    0    1    0  965    0    4    1    0   10]
 [   1    0    1    5    0  883    1    0    0    1]
 [   6    2    0    0    1    5  944    0    0    0]
 [   0    2    6    1    0    0    0 1014    1    4]
 [   3    0    2    1    1    1    0    2  959    5]
 [   4    3    0    0    3   11    0    3    0  985]]


### Precisión (Precision)

In [15]:
from sklearn.metrics import precision_score
precision_score(y_true, y_pred, average='macro')

0.9884114892277058

### Recall (Sensibilidad)

In [16]:
from sklearn.metrics import recall_score
recall_score(y_true, y_pred, average='macro')

0.9883860411320236

### F1-score

In [17]:
from sklearn.metrics import f1_score
f1_score(y_true, y_pred, average='macro')

0.9883796071680093

### Reporte de clasificación

In [18]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99       980
           1       0.99      1.00      1.00      1135
           2       0.99      0.99      0.99      1032
           3       0.99      0.99      0.99      1010
           4       0.99      0.98      0.99       982
           5       0.98      0.99      0.98       892
           6       0.99      0.99      0.99       958
           7       0.99      0.99      0.99      1028
           8       1.00      0.98      0.99       974
           9       0.98      0.98      0.98      1009

    accuracy                           0.99     10000
   macro avg       0.99      0.99      0.99     10000
weighted avg       0.99      0.99      0.99     10000



### Curva ROC y AUC

In [19]:
y_prob = model.predict(x_test)

from sklearn.metrics import roc_auc_score

roc_auc = roc_auc_score(y_true, y_prob, multi_class='ovr')
print("ROC AUC:", roc_auc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
ROC AUC: 0.9999317473167879


### Exactitud por clase

In [20]:
import numpy as np

for i in np.unique(y_true):
    idx = y_true == i
    acc = (y_pred[idx] == y_true[idx]).mean()
    print(f"Clase {i}: {acc:.3f}")

Clase 0: 0.996
Clase 1: 0.996
Clase 2: 0.993
Clase 3: 0.993
Clase 4: 0.983
Clase 5: 0.990
Clase 6: 0.985
Clase 7: 0.986
Clase 8: 0.985
Clase 9: 0.976


# AQUI